<a href="https://colab.research.google.com/github/navya039/supernan-ai-dubbing/blob/main/notebooks/colab_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Mar  7 08:27:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Installing core tools needed for video processing
!pip install gdown -q
!apt-get install -y ffmpeg -q

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [3]:
!ffmpeg -version


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [4]:
import os

!gdown "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW" -O supernan_video.mp4

# Verify download
size = os.path.getsize("supernan_video.mp4") / 1024 / 1024
print(f"✅ Video downloaded: {size:.1f} MB")

Downloading...
From (original): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW
From (redirected): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW&confirm=t&uuid=e3a08381-ff28-452e-ad17-5f410ded3e60
To: /content/supernan_video.mp4
100% 622M/622M [00:13<00:00, 44.6MB/s]
✅ Video downloaded: 593.1 MB


In [5]:
# Extracting the 0:15 to 0:30 segment from the full video
# -ss = start time, -to = end time, -c:v copy = no re-encoding (fast)
!ffmpeg -i supernan_video.mp4 -ss 00:00:15 -to 00:00:30 -c:v libx264 -c:a aac clip_15s.mp4

# Extract just the audio from the clip (needed for Whisper + voice reference)
!ffmpeg -i clip_15s.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 clip_audio.wav

# Verify both files created
clip_size = os.path.getsize("clip_15s.mp4") / 1024 / 1024
audio_size = os.path.getsize("clip_audio.wav") / 1024 / 1024
print(f" Video clip: {clip_size:.1f} MB")
print(f" Audio extracted: {audio_size:.1f} MB")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [6]:
# Installing Whisper - using large-v3 for best accuracy
!pip install openai-whisper -q
!pip install torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 29.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
import whisper

# Loading large-v3 - most accurate model, worth the extra load time
model = whisper.load_model("large-v3")

# Transcribe with word-level timestamps - this is key for chunk alignment later
result = model.transcribe(
    "clip_audio.wav",
    language="en",
    word_timestamps=True,
    verbose=False
)

# Print full transcript
print("📝 Full Transcript:")
print(result["text"])

print("\n⏱️ Word-level timestamps:")
for segment in result["segments"]:
    for word in segment["words"]:
        print(f"  {word['start']:.2f}s → {word['end']:.2f}s : {word['word']}")

100%|██████████████████████████████████████| 2.88G/2.88G [00:21<00:00, 147MiB/s]
100%|██████████| 1501/1501 [00:09<00:00, 152.14frames/s]

📝 Full Transcript:
 So now let's see how to protect your hygiene First step is to brush your teeth before booking

⏱️ Word-level timestamps:
  0.00s → 1.12s :  So
  1.12s → 1.32s :  now
  1.32s → 1.74s :  let's
  1.74s → 1.88s :  see
  1.88s → 2.02s :  how
  2.02s → 2.98s :  to
  2.98s → 3.20s :  protect
  3.20s → 3.58s :  your
  3.58s → 3.58s :  hygiene
  3.58s → 5.00s :  First
  5.00s → 6.22s :  step
  6.22s → 6.40s :  is
  6.40s → 6.90s :  to
  6.90s → 9.48s :  brush
  9.48s → 9.74s :  your
  9.74s → 10.22s :  teeth
  10.22s → 14.36s :  before
  14.36s → 14.98s :  booking


In [8]:
import json

# Saving full segment data - we'll need timestamps during TTS alignment
transcript_data = {
    "full_text": result["text"],
    "segments": result["segments"],
    "language": result["language"]
}

with open("transcript.json", "w") as f:
    json.dump(transcript_data, f, indent=2)

# Also save clean text with manual fix applied
clean_text = result["text"].replace("booking", "looking")
with open("transcript_clean.txt", "w") as f:
    f.write(clean_text)

print("✅ Transcript saved!")
print(f"📝 Clean text: {clean_text}")

✅ Transcript saved!
📝 Clean text:  So now let's see how to protect your hygiene First step is to brush your teeth before looking


In [1]:
# IndicTrans2 is built by AI4Bharat specifically for Indian languages
# Much better than Google Translate for natural Hindi
!pip install sacremoses sentencepiece -q
!pip install git+https://github.com/VarunGumma/IndicTransTokenizer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 22.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 15.2 MB/s eta 0:00:00


In [8]:
!pip install transformers sentencepiece sacremoses -q

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Switching to NLLB - no login required, excellent Hindi support
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model_trans = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

print("✅ NLLB Translation model loaded!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB Translation model loaded!


In [12]:
# Recreating transcript manually since session reset
clean_text = "So now let's see how to protect your hygiene. First step is to brush your teeth before looking"

with open("transcript_clean.txt", "w") as f:
    f.write(clean_text)

print("✅ Transcript recreated!")
print(f"📝 Text: {clean_text}")

✅ Transcript recreated!
📝 Text: So now let's see how to protect your hygiene. First step is to brush your teeth before looking


In [13]:
with open("transcript_clean.txt", "r") as f:
    english_text = f.read().strip()

print(f"📝 English: {english_text}")

inputs = tokenizer(
    english_text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# eng_Latn = English, hin_Deva = Hindi
translated = model_trans.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("hin_Deva"),
    max_length=256,
    num_beams=5
)

hindi_translation = tokenizer.decode(translated[0], skip_special_tokens=True)

print(f"🇮🇳 Hindi: {hindi_translation}")

with open("translation_hindi.txt", "w", encoding="utf-8") as f:
    f.write(hindi_translation)

print("✅ Translation saved!")

📝 English: So now let's see how to protect your hygiene. First step is to brush your teeth before looking
🇮🇳 Hindi: तो अब आइए देखें कि आप अपनी स्वच्छता की रक्षा कैसे कर सकते हैं। पहला कदम देखने से पहले अपने दांतों को ब्रश करना है।
✅ Translation saved!


In [6]:
# Manual refinement - making it natural for a nanny audience
# Original had "before looking" which doesn't make sense in context
hindi_refined = "तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।"
!pip install gTTS pydub -q
with open("translation_hindi.txt", "w", encoding="utf-8") as f:
    f.write(hindi_refined)

print("✅ Refined translation saved!")
print(f"🇮🇳 Final Hindi: {hindi_refined}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
✅ Refined translation saved!
🇮🇳 Final Hindi: तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।


## Stage 4: Voice Cloning with Coqui XTTS-v2
Cloning the original speaker's voice and generating Hindi audio.
Reference audio is extracted from the original English clip.

In [7]:
# Switching to Bark - works on Python 3.12, supports Hindi, free and open source
!pip install git+https://github.com/suno-ai/bark.git -q
!pip install scipy -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.


In [8]:
from gtts import gTTS
import os

with open("translation_hindi.txt", "r", encoding="utf-8") as f:
    hindi_text = f.read().strip()

print(f"🇮🇳 Generating audio for: {hindi_text}")

# Generate Hindi speech
tts = gTTS(text=hindi_text, lang="hi", slow=False)
tts.save("hindi_audio.mp3")

# Convert to wav for processing
!ffmpeg -i hindi_audio.mp3 -ar 16000 -ac 1 hindi_audio.wav -y -q 0

size = os.path.getsize("hindi_audio.wav") / 1024
print(f"✅ Hindi audio generated: {size:.1f} KB")

🇮🇳 Generating audio for: तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsr

In [9]:
from IPython.display import Audio
Audio("hindi_audio.wav")

In [11]:
import os

# Recreate transcript
clean_text = "So now let's see how to protect your hygiene. First step is to brush your teeth before looking"
with open("transcript_clean.txt", "w") as f:
    f.write(clean_text)

# Recreate Hindi translation
hindi_refined = "तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।"
with open("translation_hindi.txt", "w", encoding="utf-8") as f:
    f.write(hindi_refined)

print("✅ Text files recreated")

# Re-download video
!gdown "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW" -O supernan_video.mp4

# Re-extract clip and audio
!ffmpeg -i supernan_video.mp4 -ss 00:00:15 -to 00:00:30 -c:v libx264 -c:a aac clip_15s.mp4 -y
!ffmpeg -i clip_15s.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 clip_audio.wav -y

# Re-generate Hindi audio
from gtts import gTTS
tts = gTTS(text=hindi_refined, lang="hi", slow=False)
tts.save("hindi_audio.mp3")
!ffmpeg -i hindi_audio.mp3 -ar 16000 -ac 1 hindi_audio.wav -y

print("✅ All files recreated!")
!ls -lh *.wav *.mp4

✅ Text files recreated
Downloading...
From (original): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW
From (redirected): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW&confirm=t&uuid=f710eed8-8c83-45f6-9f31-06896f772734
To: /content/supernan_video.mp4
100% 622M/622M [00:10<00:00, 58.2MB/s]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmyso

In [12]:
import librosa
import soundfile as sf

original, sr_orig = librosa.load("clip_audio.wav", sr=None)
hindi, sr_hindi = librosa.load("hindi_audio.wav", sr=None)

orig_duration = librosa.get_duration(y=original, sr=sr_orig)
hindi_duration = librosa.get_duration(y=hindi, sr=sr_hindi)

print(f"⏱️ Original duration: {orig_duration:.2f}s")
print(f"⏱️ Hindi duration: {hindi_duration:.2f}s")

ratio = orig_duration / hindi_duration
hindi_aligned = librosa.effects.time_stretch(hindi, rate=ratio)

sf.write("hindi_audio_aligned.wav", hindi_aligned, sr_hindi)
print(f"✅ Audio aligned to {orig_duration:.2f}s")

⏱️ Original duration: 15.02s
⏱️ Hindi duration: 7.92s
✅ Audio aligned to 15.02s
